# 🍎 Transfer Learning with MobileNetV2 — Fruit Classifier

This notebook walks through:
1. **Concepts** — What is transfer learning?
2. **Architecture** — MobileNetV2 + custom head
3. **Training** — Phase 1 (frozen base) → Phase 2 (fine-tune)
4. **Evaluation** — Metrics, confusion matrix, sample predictions
5. **Inference** — Predict on your own images


## 1 — Concepts

### What is Transfer Learning?
Instead of training from scratch (which requires millions of images), we **reuse** a network already trained on ImageNet (1.2M images, 1000 classes).

```
ImageNet pre-trained weights
        │
        ▼
  MobileNetV2 backbone  ← frozen in Phase 1
        │
        ▼
  Custom classifier head  ← trained first
        │
        ▼
  Fine-tune top layers   ← Phase 2 (small lr)
```

### Why MobileNetV2?
| Property | Value |
|---|---|
| Parameters | ~3.4M |
| Top-1 ImageNet acc | 71.8% |
| Speed | Very fast (mobile-optimized) |
| Input size | 224×224×3 |

In [ ]:
# ── Setup ──────────────────────────────────────────────────────
import os, json, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices("GPU")))

# Config
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
CLASSES    = ['apple', 'banana', 'orange']
DATA_DIR   = 'data'

## 2 — Explore the Dataset

In [ ]:
from utils.prepare_dataset import dataset_stats
dataset_stats(DATA_DIR)

In [ ]:
# Visualise sample images per class
from tensorflow.keras.preprocessing import image as kimage

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for row, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATA_DIR, 'train', cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.png'))][:4]
    for col, fname in enumerate(imgs):
        img = kimage.load_img(os.path.join(cls_dir, fname), target_size=IMG_SIZE)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        axes[row, col].set_title(cls.capitalize() if col == 0 else '', fontweight='bold')
plt.suptitle('Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3 — Build the Model

In [ ]:
# Load MobileNetV2 without the top classification layers
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

print(f'Base model layers : {len(base_model.layers)}')
print(f'Base model params : {base_model.count_params():,}')

# Build our classifier on top
inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(CLASSES), activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 4 — Visualise MobileNet Feature Maps

In [ ]:
# Show what MobileNet 'sees' in early vs late layers
import random

feature_model = Model(
    inputs=base_model.input,
    outputs=[base_model.layers[10].output,   # Early layer
             base_model.layers[80].output,   # Mid layer
             base_model.layers[-1].output]   # Final layer
)

# Pick a random training image
sample_class = CLASSES[0]
sample_dir   = os.path.join(DATA_DIR, 'train', sample_class)
sample_file  = random.choice(os.listdir(sample_dir))
sample_img   = kimage.load_img(os.path.join(sample_dir, sample_file), target_size=IMG_SIZE)
sample_arr   = np.expand_dims(kimage.img_to_array(sample_img) / 255.0, 0)

feats = feature_model.predict(sample_arr, verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(16, 7))
titles = ['Layer 10 (edges)', 'Layer 80 (textures)', 'Final layer (semantics)']

for row, (feat_map, title) in enumerate(zip(feats, titles)):
    for col in range(8):
        ch = feat_map[0, :, :, col]
        axes[row, col].imshow(ch, cmap='viridis')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(title, fontsize=9, fontweight='bold')

plt.suptitle(f'Feature Maps — {sample_class}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 — Train & Evaluate (use train.py for full training)

In [ ]:
# Run this cell to train (or run `python train.py` in terminal)
# For a quick demo, reduce epochs:
# !python train.py   # <-- full training

print('Run in terminal: python train.py')
print('Or run evaluate.py after training: python evaluate.py')

## 6 — Load & Predict

In [ ]:
# Load trained model and predict on an image
MODEL_PATH = 'models/fruit_classifier_final.keras'

if os.path.exists(MODEL_PATH):
    trained_model = tf.keras.models.load_model(MODEL_PATH)
    with open('models/class_map.json') as f:
        class_map = {int(k): v for k, v in json.load(f).items()}

    # Predict on a validation image
    test_class = CLASSES[0]
    test_dir   = os.path.join(DATA_DIR, 'val', test_class)
    test_file  = os.listdir(test_dir)[0]
    test_path  = os.path.join(test_dir, test_file)

    img = kimage.load_img(test_path, target_size=IMG_SIZE)
    arr = np.expand_dims(kimage.img_to_array(img) / 255.0, 0)
    probs = trained_model.predict(arr, verbose=0)[0]

    print(f'Image: {test_path}  (true class: {test_class})')
    for i, p in sorted(enumerate(probs), key=lambda x: -x[1]):
        bar = '█' * int(p * 30)
        print(f'  {class_map[i]:<12} {bar} {p*100:.1f}%')
else:
    print(f'Model not found at {MODEL_PATH}. Train first with python train.py')